# SemiFA — Reproducibility Notebook

**Author:** Shivam Chand Kaushik
**Affiliation:** School of AI & Data Science, IIT Jodhpur
**Paper:** *SemiFA: An Agentic Multi-Modal Framework for Autonomous Semiconductor Failure Analysis Report Generation*
**Dataset:** [ShivamChand/SemiFA-930](https://huggingface.co/datasets/ShivamChand/SemiFA-930)
**Code:** [github.com/Shivamckaushik/SemiFA](https://github.com/Shivamckaushik/SemiFA)

---

## What this notebook reproduces

| Step | Paper Claim | Where in Paper |
|------|-------------|----------------|
| **1 — DINOv2** | 90.0% accuracy, 0.898 Macro F1 on 140-image val set | Table III |
| **3 — Latency** | Pipeline ~48 s total on A100 (per-node breakdown) | Table IV |
| **4 — FA Report** | End-to-end FA report from one image (with inspection image embedded) | Fig. 2 |
| **4b — Batch** | FA pipeline on 6 varied-class images with combined findings report | Section IV |
| **5 — QLoRA** *(optional)* | Loss collapse 0.21 → <0.001 on 790 samples (overfitting) | Section IV-B |

## Requirements
- **Runtime:** GPU → *Runtime > Change runtime type* → **A100** (Colab Pro) or T4 (free tier, ~3× slower)
- **Google Drive:** ~15 GB free space
- **HuggingFace token:** Required for LLaVA-1.6 (gated). Get yours at https://huggingface.co/settings/tokens

## One-time dataset preparation (run on your local machine once)
```bash
tar -czf semifa_dataset.tar.gz \
    data/processed \
    data/synthetic_dataset/images \
    data/wm811k/images \
    data/mixedwm38/images \
    training/qlora_finetune.py
```
Upload `semifa_dataset.tar.gz` to **MyDrive/SemiFA/** on Google Drive, then run this notebook top-to-bottom.


In [ ]:
import os, subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
if torch.cuda.is_available():
    GPU  = torch.cuda.get_device_name(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU      : {GPU}')
    print(f'VRAM     : {VRAM:.0f} GB')
else:
    GPU = 'cpu'
    print('WARNING: No GPU — Steps 1–5 require a GPU.')

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'

PROJECT_DIR  = '/content/drive/MyDrive/SemiFA'
TRAIN_JSONL  = f'{PROJECT_DIR}/data/processed/train.jsonl'
VAL_JSONL    = f'{PROJECT_DIR}/data/processed/val.jsonl'
MODELS_DIR   = f'{PROJECT_DIR}/models'
REPORTS_DIR  = f'{PROJECT_DIR}/reports/output'
HEAD_SAVE    = f'{MODELS_DIR}/dinov2_head.pt'
QLORA_DIR    = f'{MODELS_DIR}/llava-semiconductor-qlora'

DEFECT_CLASSES = [
    'scratch', 'particle_contamination', 'edge_crack',
    'center_cluster', 'local_cluster', 'ring_pattern',
    'random_defects', 'near_full_wafer', 'no_defect',
]
CLASS2IDX = {c: i for i, c in enumerate(DEFECT_CLASSES)}

print(f'\nDEVICE      : {DEVICE}')
print(f'PROJECT_DIR : {PROJECT_DIR}')
print(f'TF32        : enabled')


## Step 0 — Setup: Mount Drive, Extract Dataset, Install Packages

Run **Cells 3, 4, 5** in order. Cell 5 auto-restarts the runtime — after restart, re-run Cell 1 then continue from Cell 6.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

ARCHIVE = f'{PROJECT_DIR}/semifa_dataset.tar.gz'
MARKER  = f'{PROJECT_DIR}/.extracted'

if os.path.exists(MARKER):
    print('Dataset already extracted — skipping.')
elif os.path.exists(ARCHIVE):
    print(f'Extracting {os.path.basename(ARCHIVE)} ...')
    ret = os.system(f'tar -xzf "{ARCHIVE}" -C "{PROJECT_DIR}"')
    if ret == 0:
        open(MARKER, 'w').close()
        print('Extraction complete.')
    else:
        print('ERROR: extraction failed.')
else:
    print(f'ERROR: {ARCHIVE} not found.')
    print('Upload semifa_dataset.tar.gz to MyDrive/SemiFA/ first.')

for path, expected in [(TRAIN_JSONL, 790), (VAL_JSONL, 140)]:
    if os.path.exists(path):
        n = sum(1 for _ in open(path))
        ok = 'OK' if n == expected else f'WARNING: expected {expected}'
        print(f'{os.path.basename(path)}: {n} records [{ok}]')
    else:
        print(f'MISSING: {path}')


In [ ]:
try:
    import bitsandbytes, peft, qdrant_client, reportlab
    print('All packages already installed.')
except ImportError:
    print('Installing dependencies (~3 min)...')
    import subprocess
    subprocess.run([
        'pip', 'install', '-q',
        'transformers>=4.41.0', 'peft>=0.10.0', 'bitsandbytes>=0.46.1',
        'accelerate>=0.28.0', 'qdrant-client>=1.9.1',
        'scikit-learn>=1.4', 'reportlab>=4.1', 'Pillow',
    ], check=False)
    print('Restarting runtime...')
    import time; time.sleep(1)
    import os; os.kill(os.getpid(), 9)


## Step 1 — DINOv2 Classifier Training & Evaluation

**Reproduces: Table III — Visual Encoder Comparison**

Trains a 214K-param MLP head on frozen DINOv2-base (768-d CLS embeddings) on SemiFA-930 train split (790 images, 9 classes).

**Expected output:**
```
Overall accuracy: ~90.0%  (paper: 90.0%)
Macro F1:         ~0.898
```
Variance of ±2% across runs is normal. If `dinov2_head.pt` exists on Drive, training is skipped.

**Runtime:** A100 ~5 min | T4 ~12 min


In [ ]:
import json, time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
from sklearn.metrics import classification_report

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

DINOV2_ID  = 'facebook/dinov2-base'
IMAGE_ROOT = Path(PROJECT_DIR)

def load_records(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

class WaferDS(torch.utils.data.Dataset):
    def __init__(self, records, root):
        self.items = [
            (root / r['image'], CLASS2IDX[r['defect_class']])
            for r in records
            if (root / r['image']).exists() and r.get('defect_class') in CLASS2IDX
        ]
        print(f'  {len(self.items)} images')
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, lbl = self.items[i]
        return Image.open(p).convert('RGB'), lbl

class MLPHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 9)
        )
    def forward(self, x): return self.net(x)

def extract_features(ds, bbone, proc, bs=128):
    def collate(batch):
        imgs, lbls = zip(*batch)
        return list(imgs), list(lbls)
    dl = DataLoader(ds, batch_size=bs, shuffle=False,
                    num_workers=4, pin_memory=True, collate_fn=collate)
    feats, labels = [], []
    for imgs, lbls in dl:
        inp = proc(images=imgs, return_tensors='pt')
        inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            out = bbone(**inp)
        feats.append(out.last_hidden_state[:, 0, :].float().cpu())
        labels.extend(lbls)
    return torch.cat(feats), torch.tensor(labels)

# ── Load backbone ─────────────────────────────────────────────────────────────
print('[1/3] Loading DINOv2-base (frozen) ...')
proc_dino = AutoImageProcessor.from_pretrained(DINOV2_ID)
backbone  = AutoModel.from_pretrained(DINOV2_ID).to(DEVICE).eval()
for p in backbone.parameters():
    p.requires_grad = False
print(f'      Backbone: {sum(p.numel() for p in backbone.parameters())/1e6:.0f}M params (frozen)')

head = MLPHead().to(DEVICE)
print(f'      MLP head: {sum(p.numel() for p in head.parameters()):,} params (trainable)')

# ── Load from checkpoint or train ────────────────────────────────────────────
import os
if os.path.exists(HEAD_SAVE):
    print(f'\nCheckpoint found: {HEAD_SAVE}')
    ckpt = torch.load(HEAD_SAVE, map_location=DEVICE)
    head.load_state_dict(ckpt['head'] if isinstance(ckpt, dict) else ckpt)
    print('Weights loaded — skipping training.')
    skip_train = True
else:
    skip_train = False

if not skip_train:
    print('\n[2/3] Extracting DINOv2 features ...')
    train_recs = load_records(TRAIN_JSONL)
    val_recs   = load_records(VAL_JSONL)
    print('  Train:', end=' '); train_ds = WaferDS(train_recs, IMAGE_ROOT)
    print('  Val:',   end=' '); val_ds   = WaferDS(val_recs,   IMAGE_ROOT)
    t0 = time.time()
    X_tr, y_tr = extract_features(train_ds, backbone, proc_dino)
    X_va, y_va = extract_features(val_ds,   backbone, proc_dino)
    print(f'  {time.time()-t0:.1f}s | train={tuple(X_tr.shape)} val={tuple(X_va.shape)}')

    print('\n[3/3] Training MLP head (50 epochs) ...')
    Xtr = X_tr.to(DEVICE); ytr = y_tr.to(DEVICE)
    Xva = X_va.to(DEVICE); yva = y_va.to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss(); BS = 512

    best_acc, best_state = 0.0, None
    t0 = time.time()
    for epoch in range(50):
        head.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), BS):
            idx = perm[i:i+BS]
            opt.zero_grad(); crit(head(Xtr[idx]), ytr[idx]).backward(); opt.step()
        head.eval()
        with torch.no_grad():
            acc = (head(Xva).argmax(1) == yva).float().mean().item()
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.clone() for k, v in head.state_dict().items()}
        if (epoch + 1) % 10 == 0:
            print(f'  epoch {epoch+1:3d}/50  val_acc={acc*100:.1f}%')
    print(f'  Done {time.time()-t0:.1f}s | best_val_acc={best_acc*100:.1f}%')
    head.load_state_dict(best_state)
    torch.save({'head': best_state, 'classes': DEFECT_CLASSES}, HEAD_SAVE)
    print(f'  Saved: {HEAD_SAVE}')

# ── Evaluate ──────────────────────────────────────────────────────────────────
print('\n-- Evaluation on SemiFA-930 val set (n=140) --')
head.eval()
val_recs_eval = load_records(VAL_JSONL)
val_ds_eval   = WaferDS(val_recs_eval, IMAGE_ROOT)
X_va_eval, y_va_eval = extract_features(val_ds_eval, backbone, proc_dino)

with torch.no_grad():
    preds = head(X_va_eval.to(DEVICE)).argmax(1).cpu().numpy()
truth = y_va_eval.numpy()
acc = (preds == truth).mean() * 100
print(f'\nOverall accuracy : {(preds==truth).sum()}/{len(truth)} = {acc:.1f}%  (paper: 90.0%)')
print()
print(classification_report(truth, preds, target_names=DEFECT_CLASSES, digits=3))


## Step 2 — Load LLaVA-1.6 Base Model

Loads `llava-hf/llava-v1.6-mistral-7b-hf` in 4-bit NF4. Downloads ~4 GB on first run.

Flash Attention 2 is attempted; falls back to standard attention automatically.
Base model is used — QLoRA adapter overfit on 790 samples (see Step 5).

**Runtime:** ~3 min first run, ~1 min cache hit.


In [ ]:
import os, torch
from getpass import getpass
from transformers import (
    LlavaNextForConditionalGeneration, LlavaNextProcessor, BitsAndBytesConfig
)

HF_TOKEN = os.environ.get('HF_TOKEN', '') or getpass('HuggingFace token (hf_...): ')
os.environ['HF_TOKEN'] = HF_TOKEN

BASE_MODEL_ID = 'llava-hf/llava-v1.6-mistral-7b-hf'
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

print(f'Loading {BASE_MODEL_ID} (4-bit NF4) ...')
processor = LlavaNextProcessor.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, use_fast=False)
try:
    model = LlavaNextForConditionalGeneration.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb_cfg, device_map='auto',
        attn_implementation='flash_attention_2', token=HF_TOKEN,
    )
    print('  Attention: Flash Attention 2')
except Exception as e:
    print(f'  Flash Attention 2 unavailable ({type(e).__name__}) — standard attention')
    model = LlavaNextForConditionalGeneration.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb_cfg, device_map='auto', token=HF_TOKEN,
    )
model.eval()
print(f'Model ready.  VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated')

SYS_PROMPT = (
    'You are an expert semiconductor failure analysis engineer. '
    'Analyse inspection images and answer questions accurately and concisely.'
)

def llava_infer(image, question, max_new_tokens=350):
    """Single LLaVA-1.6 inference call; returns decoded text."""
    messages = [{'role': 'user', 'content': [
        {'type': 'image'}, {'type': 'text', 'text': question}
    ]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    if '[INST]' in prompt and '<<SYS>>' not in prompt:
        prompt = prompt.replace('[INST]',
            f'[INST] <<SYS>>\n{SYS_PROMPT}\n<</SYS>>\n\n', 1)
    inputs = processor(images=image, text=prompt, return_tensors='pt')
    inputs = {
        k: v.to(DEVICE, dtype=torch.bfloat16) if v.dtype == torch.float32 else v.to(DEVICE)
        for k, v in inputs.items()
    }
    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.3,
            repetition_penalty=1.3, no_repeat_ngram_size=4,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    response = processor.decode(out[0], skip_special_tokens=True)
    torch.cuda.empty_cache()
    return response.split('[/INST]')[-1].strip()

print('llava_infer() ready.')


## Step 3 — Pipeline Latency Benchmark

**Reproduces: Table IV — Pipeline Execution Time**

**Expected (A100 SXM4-40GB):**
```
DefectDescriber    22.0s
RootCauseAnalyzer  11.9s
SeverityClassifier  5.5s
RecipeAdvisor       6.5s
Total             ~48.4s
```
Variance ±20% is normal. Requires Step 2.


In [ ]:
import time, statistics, json
from pathlib import Path
from PIL import Image
import numpy as np

val_recs = [json.loads(l) for l in open(VAL_JSONL) if l.strip()]
sample_path = Path(PROJECT_DIR) / val_recs[0]['image']
bench_img = (Image.open(sample_path).convert('RGB')
             if sample_path.exists()
             else Image.fromarray(np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)))
print(f'Benchmark image: {sample_path.name}')

BENCH_NODES = [
    ('DefectDescriber',
     'This semiconductor inspection image shows a scratch defect. '
     'Describe the defect morphology, spatial distribution, and affected die percentage.',
     200),
    ('RootCauseAnalyzer',
     'Defect: scratch (83%). Equipment: vacuum_level=448 mbar (threshold 500) at T-45 min. '
     'Generate 3 ranked root cause hypotheses referencing specific equipment parameters.',
     350),
    ('SeverityClassifier',
     'Classify severity (CRITICAL/MAJOR/MINOR/NONE) of this scratch defect. '
     'Estimate yield impact %. Justify in one sentence.',
     150),
    ('RecipeAdvisor',
     'Defect: scratch. Severity: MAJOR. Root cause: vacuum chuck pressure drop. '
     'Provide 3 corrective actions with specific process parameter targets.',
     250),
]

print('\nTiming — median of 3 runs per node ...')
print('=' * 52)
total = 0.0
for name, prompt, max_tok in BENCH_NODES:
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        llava_infer(bench_img, prompt, max_new_tokens=max_tok)
        times.append(time.perf_counter() - t0)
    med = statistics.median(times)
    total += med
    print(f'{name:<24} {med:5.1f}s')
print(f'{"ReportGenerator":<24}  ~2.5s  (ReportLab PDF)')
print('=' * 52)
print(f'{"Total":<24} {total:.1f}s + ~2.5s')
print(f'\nPaper baseline (A100):  48.4s')


## Step 4 — Full FA Pipeline: Single Image

**Reproduces: Fig. 2 — End-to-End FA Report**

Runs the complete 5-node pipeline on one image and generates a PDF FA report with the inspection image embedded.

Nodes: DefectDescriber → RootCauseAnalyzer → SeverityClassifier → RecipeAdvisor → ReportGenerator

**Expected time:** ~48 s on A100. Requires Steps 1 and 2.


In [ ]:
import re, uuid, json, time, os, io
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
import torch, numpy as np

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable, Image as RLImage,
)

# ── Qdrant in-memory ──────────────────────────────────────────────────────────
qdrant = QdrantClient(':memory:')
qdrant.create_collection('defect_history',
    vectors_config=VectorParams(size=768, distance=Distance.COSINE))
rng = np.random.default_rng(42)
for idx, cls in enumerate(DEFECT_CLASSES):
    for k in range(5):
        vec = rng.normal(0, 1, 768); vec = (vec / np.linalg.norm(vec)).tolist()
        qdrant.upsert('defect_history', points=[PointStruct(
            id=idx*10+k, vector=vec,
            payload={'defect_class': cls, 'equipment_id': 'EQ-INSP-01',
                     'severity': 'MAJOR', 'lot_id': f'LOT-HIST-{idx:03d}'})])
print(f'Qdrant: {qdrant.count("defect_history").count} records loaded')

# ── Select image ──────────────────────────────────────────────────────────────
import random; random.seed(42)
val_recs = [json.loads(l) for l in open(VAL_JSONL) if l.strip()]
sample   = random.choice(val_recs)
img_path = Path(PROJECT_DIR) / sample['image']
image    = Image.open(img_path).convert('RGB')
print(f'Image : {img_path.name}  (true label={sample.get("defect_class","unknown")})')
print()

# ── Pipeline ──────────────────────────────────────────────────────────────────
t_start = time.perf_counter()

# Node 1: DefectDescriber
print('[1/5] DefectDescriber ...', end=' ', flush=True)
t0 = time.perf_counter()
dino_inp = proc_dino(images=[image], return_tensors='pt')
dino_inp = {k: v.to(DEVICE) for k, v in dino_inp.items()}
with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
    emb   = backbone(**dino_inp).last_hidden_state[:, 0, :].float()
    probs = torch.softmax(head(emb), dim=1)[0]
pred_idx   = probs.argmax().item()
pred_class = DEFECT_CLASSES[pred_idx]
pred_conf  = probs[pred_idx].item()
emb_vec    = emb[0].cpu().numpy().tolist()
defect_desc = llava_infer(image,
    f'This semiconductor inspection image contains a {pred_class.replace("_"," ")} defect '
    f'(DINOv2 confidence {pred_conf:.1%}). Describe the defect morphology, spatial '
    'distribution, and affected area using precise failure analysis terminology.',
    max_new_tokens=200)
print(f'{time.perf_counter()-t0:.1f}s | class={pred_class} conf={pred_conf:.1%}')

# Node 2: RootCauseAnalyzer
print('[2/5] RootCauseAnalyzer ...', end=' ', flush=True)
t0 = time.perf_counter()
SIM_TELEMETRY = (
    'SECS/GEM S5F1 Alarm 0x0042: vacuum_level=448 mbar (threshold 500 mbar) at T-45 min. '
    'S6F11 CEID 1012: handler_speed=322 mm/s (limit 300 mm/s) at T-30 min. '
    'chuck_temp=24.1 C (nominal 23+/-2 C). bond_force=18.4 g (nominal).'
)
similar = qdrant.query_points('defect_history', query=emb_vec, limit=5).points
hist_ctx = ', '.join(f'{h.payload["defect_class"]}({h.payload["lot_id"]})' for h in similar)
rca_text = llava_infer(image,
    f'Defect: {pred_class.replace("_"," ")} (confidence {pred_conf:.0%}).\n'
    f'Equipment telemetry: {SIM_TELEMETRY}\n'
    f'Similar historical defects: {hist_ctx}\n\n'
    'Provide exactly 3 numbered root cause hypotheses, each referencing specific equipment parameters.',
    max_new_tokens=350)
hypotheses = [l.strip() for l in rca_text.split('\n')
              if l.strip() and len(l.strip())>20 and l.strip()[0].isdigit()][:3] or [rca_text[:300]]
print(f'{time.perf_counter()-t0:.1f}s | {len(hypotheses)} hypotheses')

# Node 3: SeverityClassifier
print('[3/5] SeverityClassifier ...', end=' ', flush=True)
t0 = time.perf_counter()
sev_text = llava_infer(image,
    f'Classify severity of this {pred_class.replace("_"," ")} defect.\n'
    'Format exactly:\nSEVERITY: <CRITICAL|MAJOR|MINOR|NONE>\n'
    'YIELD_IMPACT: <N>%\nREASONING: <one sentence>',
    max_new_tokens=150)
severity = 'MINOR'; yield_pct = 2.0; sev_reason = sev_text.strip()
for ln in sev_text.split('\n'):
    for s in ('CRITICAL','MAJOR','MINOR','NONE'):
        if s in ln.upper(): severity = s
    m = re.search(r'(\d+\.?\d*)\s*%', ln)
    if m: yield_pct = float(m.group(1))
    if ln.upper().startswith('REASONING:'): sev_reason = ln.split(':',1)[-1].strip()
print(f'{time.perf_counter()-t0:.1f}s | severity={severity} yield={yield_pct}%')

# Node 4: RecipeAdvisor
print('[4/5] RecipeAdvisor ...', end=' ', flush=True)
t0 = time.perf_counter()
rec_text = llava_infer(image,
    f'Equipment: EQ-INSP-01 | Defect: {pred_class.replace("_"," ")} | Severity: {severity}\n'
    f'Root cause: {hypotheses[0][:200]}\n\n'
    'Provide exactly 3 corrective actions with specific process parameter targets. Numbered list.',
    max_new_tokens=300)
actions = [l.strip() for l in rec_text.split('\n')
           if l.strip() and len(l.strip())>15 and l.strip()[0].isdigit()][:5] or [rec_text[:300]]
print(f'{time.perf_counter()-t0:.1f}s | {len(actions)} actions')

# Node 5: ReportGenerator
print('[5/5] ReportGenerator ...', end=' ', flush=True)
t0 = time.perf_counter()

elapsed   = time.perf_counter() - t_start
report_id = uuid.uuid4().hex[:8]
now       = datetime.now(timezone.utc).isoformat()
pdf_path  = f'{REPORTS_DIR}/FA_Report_{report_id}.pdf'
os.makedirs(REPORTS_DIR, exist_ok=True)

ss   = getSampleStyleSheet()
h1   = ParagraphStyle('H1', parent=ss['Title'],   fontSize=18, spaceAfter=4)
h2   = ParagraphStyle('H2', parent=ss['Heading2'],fontSize=12, spaceBefore=10, spaceAfter=4,
                       textColor=colors.HexColor('#2C3E50'))
body = ParagraphStyle('B', parent=ss['Normal'],   fontSize=10, leading=14)
foot = ParagraphStyle('F', parent=ss['Normal'],   fontSize=8,  textColor=colors.grey)
SEV_COLOURS = {'CRITICAL': colors.HexColor('#C0392B'), 'MAJOR': colors.HexColor('#E67E22'),
               'MINOR':    colors.HexColor('#F1C40F'),  'NONE':  colors.HexColor('#27AE60')}
sev_colour = SEV_COLOURS.get(severity, colors.grey)

# Convert PIL image to RLImage (embed in PDF)
orig_w, orig_h = image.size
max_w = 130 * mm
rl_w  = max_w
rl_h  = rl_w * (orig_h / orig_w)
img_buf = io.BytesIO()
image.save(img_buf, format='PNG')
img_buf.seek(0)
rl_img = RLImage(img_buf, width=rl_w, height=rl_h)

story = [
    Paragraph('Failure Analysis Report', h1),
    Paragraph(f'Report ID: {report_id} | Generated: {now}', ss['Normal']),
    Spacer(1, 4*mm),
    HRFlowable(width='100%', thickness=1, color=colors.HexColor('#2C3E50')),
    Spacer(1, 4*mm),

    # Header table
    Paragraph('Inspection Details', h2),
    Table(
        [['Equipment', 'EQ-INSP-01', 'Lot ID', 'LOT-2024-COLAB'],
         ['Wafer',     'W05',        'Modality', 'sem']],
        colWidths=[35*mm, 55*mm, 25*mm, 55*mm],
        style=TableStyle([
            ('GRID',       (0,0),(-1,-1), 0.5, colors.HexColor('#BDC3C7')),
            ('BACKGROUND', (0,0),(0,-1),  colors.HexColor('#ECF0F1')),
            ('BACKGROUND', (2,0),(2,-1),  colors.HexColor('#ECF0F1')),
            ('FONTNAME',   (0,0),(-1,-1), 'Helvetica'),
            ('FONTSIZE',   (0,0),(-1,-1), 9),
            ('PADDING',    (0,0),(-1,-1), 4),
        ])
    ),
    Spacer(1, 4*mm),

    # Severity banner
    Paragraph('Severity Assessment', h2),
    Table([[
        Paragraph(f'<b>SEVERITY: {severity}</b>', body),
        Paragraph(f'Yield Impact: <b>{yield_pct:.1f}%</b>', body),
    ]], colWidths=[85*mm, 85*mm],
    style=TableStyle([
        ('BACKGROUND', (0,0),(0,0), sev_colour),
        ('TEXTCOLOR',  (0,0),(0,0), colors.white),
        ('BACKGROUND', (1,0),(1,0), colors.HexColor('#F8F9FA')),
        ('GRID',       (0,0),(-1,-1), 0.5, colors.HexColor('#BDC3C7')),
        ('PADDING',    (0,0),(-1,-1), 6),
        ('FONTNAME',   (0,0),(-1,-1), 'Helvetica-Bold'),
    ])),
    Paragraph(sev_reason[:400], body),
    Spacer(1, 3*mm),

    # Inspection image embedded
    Paragraph('Inspection Image', h2),
    Paragraph(f'File: {img_path.name}  |  True label: {sample.get("defect_class","unknown")}', body),
    Spacer(1, 2*mm),
    rl_img,
    Spacer(1, 3*mm),

    # Defect description
    Paragraph('Defect Description', h2),
    Paragraph(f'<b>Class:</b> {pred_class} (DINOv2 confidence: {pred_conf:.1%})', body),
    Spacer(1, 2*mm),
    Paragraph(defect_desc[:1500], body),
    Spacer(1, 3*mm),

    # Root cause
    Paragraph('Root Cause Analysis', h2),
    Paragraph(f'<b>Equipment telemetry (representative SECS/GEM log):</b>', body),
    Paragraph(SIM_TELEMETRY, body),
    Spacer(1, 3*mm),
]
for i, hyp in enumerate(hypotheses, 1):
    story.append(Paragraph(f'{i}. {hyp[:500]}', body))
story.append(Paragraph(
    f'({len(similar)} similar defects retrieved from Qdrant vector DB)',
    ParagraphStyle('S', parent=ss['Normal'], fontSize=8, textColor=colors.grey)))

story += [
    Spacer(1, 3*mm),
    Paragraph('Corrective Action Recommendations', h2),
]
for i, act in enumerate(actions, 1):
    story.append(Paragraph(f'{i}. {act[:500]}', body))
story += [
    Spacer(1, 8*mm),
    HRFlowable(width='100%', thickness=0.5, color=colors.grey),
    Paragraph(
        f'SemiFA Autonomous FA System | Pipeline: {elapsed:.1f}s | '
        f'DINOv2-base + LLaVA-1.6 4-bit NF4 | IIT Jodhpur', foot),
]

SimpleDocTemplate(pdf_path, pagesize=A4,
    leftMargin=20*mm, rightMargin=20*mm, topMargin=20*mm, bottomMargin=20*mm,
).build(story)

print(f'{time.perf_counter()-t0:.1f}s')
print(f'\nReport ID : {report_id}')
print(f'Defect    : {pred_class} ({pred_conf:.1%})')
print(f'Severity  : {severity} | Yield impact: {yield_pct:.1f}%')
print(f'Total time: {elapsed:.1f}s  (paper: 48.4s on A100)')
print(f'PDF       : {pdf_path}')

from google.colab import files
files.download(pdf_path)
print('Download triggered.')


## Step 4b — Batch Analysis: 6 Images, Combined Findings Report

Selects **one image per defect class** (up to 6 classes) from the val set and runs the full FA pipeline on each. Produces a single **multi-image findings PDF** with:

- **Summary table** — all 6 images: filename, predicted class, confidence, severity, yield impact, time
- **Per-image sections** — embedded inspection image + classification + root cause + recommendations

This is the experimental findings documentation referenced in the paper.

**Expected time:** ~6 × 48s ≈ **5–6 min on A100** | ~25 min on T4

Requires Steps 1 and 2.


In [ ]:
import re, uuid, json, time, os, io, random
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
import torch, numpy as np

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable, PageBreak, Image as RLImage,
)

# ── Select one image per defect class (up to 6) ───────────────────────────────
val_recs = [json.loads(l) for l in open(VAL_JSONL) if l.strip()]
random.seed(7)
by_class = {}
for r in random.sample(val_recs, len(val_recs)):   # shuffle so we pick varied samples
    cls = r.get('defect_class', '')
    p   = Path(PROJECT_DIR) / r['image']
    if cls not in by_class and p.exists():
        by_class[cls] = r

selected = list(by_class.values())[:6]
print(f'Selected {len(selected)} images ({len(selected)} defect classes):')
for s in selected:
    print(f'  {s["defect_class"]:<25}  {s["image"]}')

# ── Simulated SECS/GEM telemetry (same representative log for all images) ─────
SIM_TELEMETRY = (
    'SECS/GEM S5F1 Alarm 0x0042: vacuum_level=448 mbar (threshold 500 mbar) at T-45 min. '
    'S6F11 CEID 1012: handler_speed=322 mm/s (limit 300 mm/s) at T-30 min. '
    'chuck_temp=24.1 C (nominal 23+/-2 C). bond_force=18.4 g (nominal).'
)

# ── Run full FA pipeline on each image ───────────────────────────────────────
results = []
batch_start = time.perf_counter()

for run_idx, sample in enumerate(selected):
    img_path  = Path(PROJECT_DIR) / sample['image']
    true_label = sample.get('defect_class', 'unknown')
    image     = Image.open(img_path).convert('RGB')

    print(f'\n[{run_idx+1}/{len(selected)}] {img_path.name}  (label={true_label})')
    t_start = time.perf_counter()

    # -- Node 1: DefectDescriber --
    dino_inp = proc_dino(images=[image], return_tensors='pt')
    dino_inp = {k: v.to(DEVICE) for k, v in dino_inp.items()}
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        emb   = backbone(**dino_inp).last_hidden_state[:, 0, :].float()
        probs = torch.softmax(head(emb), dim=1)[0]
    pred_idx   = probs.argmax().item()
    pred_class = DEFECT_CLASSES[pred_idx]
    pred_conf  = probs[pred_idx].item()
    emb_vec    = emb[0].cpu().numpy().tolist()

    defect_desc = llava_infer(image,
        f'This semiconductor image contains a {pred_class.replace("_"," ")} defect '
        f'(confidence {pred_conf:.1%}). Describe the morphology and spatial distribution '
        'in 3 sentences using FA terminology.',
        max_new_tokens=180)
    print(f'  Node1: {pred_class} ({pred_conf:.1%})', end='  ', flush=True)

    # -- Node 2: RootCauseAnalyzer --
    rca_text = llava_infer(image,
        f'Defect: {pred_class.replace("_"," ")} ({pred_conf:.0%}).\n'
        f'Telemetry: {SIM_TELEMETRY}\n'
        'List 2 probable root causes concisely. Numbered.',
        max_new_tokens=200)
    hypotheses = [l.strip() for l in rca_text.split('\n')
                  if l.strip() and len(l.strip())>15 and l.strip()[0].isdigit()][:2]                   or [rca_text[:200]]
    print(f'Node2: {len(hypotheses)} hyp', end='  ', flush=True)

    # -- Node 3: SeverityClassifier --
    sev_text = llava_infer(image,
        f'Classify severity of this {pred_class.replace("_"," ")} defect: '
        'CRITICAL/MAJOR/MINOR/NONE. Then yield impact %. Then one-sentence reason.',
        max_new_tokens=100)
    severity = 'MINOR'; yield_pct = 2.0; sev_reason = sev_text.strip()[:300]
    for ln in sev_text.split('\n'):
        for s in ('CRITICAL','MAJOR','MINOR','NONE'):
            if s in ln.upper(): severity = s
        m = re.search(r'(\d+\.?\d*)\s*%', ln)
        if m: yield_pct = float(m.group(1))
    print(f'Node3: {severity}', end='  ', flush=True)

    # -- Node 4: RecipeAdvisor --
    rec_text = llava_infer(image,
        f'Defect: {pred_class.replace("_"," ")} | Severity: {severity}.\n'
        f'Root cause: {hypotheses[0][:150]}\n'
        'Give 2 corrective actions with parameter targets. Numbered.',
        max_new_tokens=200)
    actions = [l.strip() for l in rec_text.split('\n')
               if l.strip() and len(l.strip())>15 and l.strip()[0].isdigit()][:3]                or [rec_text[:200]]
    print(f'Node4: {len(actions)} actions')

    elapsed = time.perf_counter() - t_start
    correct = (pred_class == true_label)
    print(f'  Time: {elapsed:.1f}s | Correct: {correct}')

    results.append({
        'run_num'    : run_idx + 1,
        'filename'   : img_path.name,
        'true_label' : true_label,
        'pred_class' : pred_class,
        'pred_conf'  : pred_conf,
        'correct'    : correct,
        'description': defect_desc,
        'hypotheses' : hypotheses,
        'severity'   : severity,
        'yield_pct'  : yield_pct,
        'sev_reason' : sev_reason,
        'actions'    : actions,
        'elapsed'    : elapsed,
        'image'      : image,
    })

batch_elapsed = time.perf_counter() - batch_start
n_correct = sum(1 for r in results if r['correct'])
print(f'\n{"="*50}')
print(f'Batch complete: {batch_elapsed:.0f}s total')
print(f'Accuracy: {n_correct}/{len(results)} = {n_correct/len(results):.1%}')

# ── Build combined findings PDF ───────────────────────────────────────────────
print('\nBuilding combined findings report ...')
os.makedirs(REPORTS_DIR, exist_ok=True)
report_id = uuid.uuid4().hex[:8]
now       = datetime.now(timezone.utc).isoformat()
pdf_path  = f'{REPORTS_DIR}/SemiFA_BatchFindings_{report_id}.pdf'

ss   = getSampleStyleSheet()
h1   = ParagraphStyle('H1', parent=ss['Title'],   fontSize=18, spaceAfter=4)
h2   = ParagraphStyle('H2', parent=ss['Heading2'],fontSize=13, spaceBefore=8, spaceAfter=3,
                       textColor=colors.HexColor('#2C3E50'))
h3   = ParagraphStyle('H3', parent=ss['Heading3'],fontSize=11, spaceBefore=6, spaceAfter=2,
                       textColor=colors.HexColor('#2980B9'))
body = ParagraphStyle('B',  parent=ss['Normal'],  fontSize=10, leading=14)
sm   = ParagraphStyle('SM', parent=ss['Normal'],  fontSize=8,  textColor=colors.grey)
SEV_COLOURS = {
    'CRITICAL': colors.HexColor('#C0392B'), 'MAJOR': colors.HexColor('#E67E22'),
    'MINOR':    colors.HexColor('#F1C40F'),  'NONE': colors.HexColor('#27AE60'),
}

story = []

# ── Cover / summary ───────────────────────────────────────────────────────────
story += [
    Paragraph('SemiFA — Multi-Image Batch Findings Report', h1),
    Paragraph(f'Generated: {now}  |  Batch ID: {report_id}', ss['Normal']),
    Paragraph(f'Images analysed: {len(results)}  |  '
              f'Classification accuracy: {n_correct}/{len(results)} ({n_correct/len(results):.1%})  |  '
              f'Total pipeline time: {batch_elapsed:.0f}s', ss['Normal']),
    Spacer(1, 4*mm),
    HRFlowable(width='100%', thickness=1, color=colors.HexColor('#2C3E50')),
    Spacer(1, 4*mm),
    Paragraph('Summary Table', h2),
]

# Summary table header + rows
tbl_data = [['#', 'Filename', 'True Label', 'Predicted', 'Conf.', 'Correct', 'Severity', 'Yield%', 'Time(s)']]
for r in results:
    tbl_data.append([
        str(r['run_num']),
        r['filename'][:28],
        r['true_label'],
        r['pred_class'],
        f"{r['pred_conf']:.1%}",
        'Y' if r['correct'] else 'N',
        r['severity'],
        f"{r['yield_pct']:.1f}",
        f"{r['elapsed']:.1f}",
    ])

# Row colours: green if correct, red if wrong
row_styles = [
    ('BACKGROUND', (0,0), (-1,0),  colors.HexColor('#2C3E50')),
    ('TEXTCOLOR',  (0,0), (-1,0),  colors.white),
    ('FONTNAME',   (0,0), (-1,0),  'Helvetica-Bold'),
    ('FONTSIZE',   (0,0), (-1,-1), 8),
    ('GRID',       (0,0), (-1,-1), 0.5, colors.HexColor('#BDC3C7')),
    ('PADDING',    (0,0), (-1,-1), 3),
    ('ALIGN',      (0,0), (-1,-1), 'CENTER'),
    ('ALIGN',      (1,0), (3,-1),  'LEFT'),
]
for i, r in enumerate(results, 1):
    bg = colors.HexColor('#D5F5E3') if r['correct'] else colors.HexColor('#FADBD8')
    row_styles.append(('BACKGROUND', (0,i), (-1,i), bg))

col_widths = [8*mm, 42*mm, 28*mm, 28*mm, 14*mm, 14*mm, 18*mm, 14*mm, 14*mm]
t = Table(tbl_data, colWidths=col_widths, style=TableStyle(row_styles))
story += [t, Spacer(1, 4*mm)]

# Methodology note
story += [
    Paragraph('Methodology', h2),
    Paragraph(
        f'Each image was processed through the SemiFA 5-node LangGraph pipeline: '
        f'(1) DefectDescriber — DINOv2-base classification + LLaVA-1.6 defect narration; '
        f'(2) RootCauseAnalyzer — simulated SECS/GEM telemetry + LLaVA root cause hypotheses; '
        f'(3) SeverityClassifier — LLaVA severity rating + yield impact estimate; '
        f'(4) RecipeAdvisor — LLaVA corrective action recommendations; '
        f'(5) ReportGenerator — ReportLab PDF assembly. '
        f'Telemetry inputs are representative SECS/GEM alarm logs (simulated). '
        f'All inference uses LLaVA-1.6 base model (4-bit NF4, no fine-tuned adapter).',
        body),
    Spacer(1, 2*mm),
]

# ── Per-image sections ─────────────────────────────────────────────────────────
for r in results:
    story.append(PageBreak())

    sev_colour = SEV_COLOURS.get(r['severity'], colors.grey)
    correct_str = 'Correct' if r['correct'] else f'Incorrect (true: {r["true_label"]})'

    story += [
        Paragraph(f'Image {r["run_num"]}: {r["filename"]}', h2),
        HRFlowable(width='100%', thickness=0.5, color=colors.HexColor('#BDC3C7')),
        Spacer(1, 3*mm),
    ]

    # Two-column: image + classification metrics
    orig_w, orig_h = r['image'].size
    thumb_w = 80 * mm
    thumb_h = thumb_w * (orig_h / orig_w)
    img_buf = io.BytesIO()
    r['image'].save(img_buf, format='PNG')
    img_buf.seek(0)
    rl_thumb = RLImage(img_buf, width=thumb_w, height=thumb_h)

    info_data = [
        ['True Label',  r['true_label']],
        ['Predicted',   r['pred_class']],
        ['Confidence',  f"{r['pred_conf']:.1%}"],
        ['Classification', correct_str],
        ['Severity',    r['severity']],
        ['Yield Impact',f"{r['yield_pct']:.1f}%"],
        ['Pipeline Time', f"{r['elapsed']:.1f}s"],
    ]
    info_tbl = Table(info_data, colWidths=[35*mm, 55*mm],
        style=TableStyle([
            ('BACKGROUND', (0,0),(0,-1), colors.HexColor('#ECF0F1')),
            ('GRID',       (0,0),(-1,-1), 0.5, colors.HexColor('#BDC3C7')),
            ('FONTSIZE',   (0,0),(-1,-1), 9),
            ('PADDING',    (0,0),(-1,-1), 4),
            ('BACKGROUND', (0,4),(1,4),   sev_colour),
            ('TEXTCOLOR',  (0,4),(1,4),   colors.white),
            ('FONTNAME',   (0,4),(1,4),   'Helvetica-Bold'),
        ]))

    # Side-by-side: thumbnail left, info table right
    side_tbl = Table([[rl_thumb, info_tbl]],
        colWidths=[thumb_w + 4*mm, 92*mm],
        style=TableStyle([
            ('VALIGN',  (0,0),(-1,-1), 'TOP'),
            ('PADDING', (0,0),(-1,-1), 2),
        ]))
    story += [side_tbl, Spacer(1, 4*mm)]

    # Severity detail
    story += [
        Paragraph('Severity Assessment', h3),
        Paragraph(r['sev_reason'][:500], body),
        Spacer(1, 3*mm),
        Paragraph('Defect Description', h3),
        Paragraph(r['description'][:800], body),
        Spacer(1, 3*mm),
        Paragraph('Root Cause Hypotheses', h3),
    ]
    for i, hyp in enumerate(r['hypotheses'], 1):
        story.append(Paragraph(f'{i}. {hyp[:400]}', body))
    story += [
        Spacer(1, 3*mm),
        Paragraph('Corrective Actions', h3),
    ]
    for i, act in enumerate(r['actions'], 1):
        story.append(Paragraph(f'{i}. {act[:400]}', body))
    story.append(Spacer(1, 2*mm))
    story.append(Paragraph(
        f'Equipment telemetry (representative SECS/GEM): {SIM_TELEMETRY}', sm))

# ── Footer ────────────────────────────────────────────────────────────────────
story += [
    Spacer(1, 6*mm),
    HRFlowable(width='100%', thickness=0.5, color=colors.grey),
    Paragraph(
        f'SemiFA Batch Analysis — {len(results)} images | {batch_elapsed:.0f}s total | '
        f'DINOv2-base + LLaVA-1.6 4-bit NF4 | IIT Jodhpur', sm),
]

SimpleDocTemplate(pdf_path, pagesize=A4,
    leftMargin=20*mm, rightMargin=20*mm, topMargin=20*mm, bottomMargin=20*mm,
).build(story)

print(f'PDF saved: {pdf_path}')
print()
print('FINDINGS SUMMARY')
print('='*55)
for r in results:
    flag = 'OK' if r['correct'] else 'MISMATCH'
    print(f'  {r["run_num"]}. {r["true_label"]:<22} -> {r["pred_class"]:<22} {r["pred_conf"]:.1%}  {r["severity"]:<8} [{flag}]')
print(f'{"="*55}')
print(f'Accuracy: {n_correct}/{len(results)}  |  Total time: {batch_elapsed:.0f}s')

from google.colab import files
files.download(pdf_path)
print('Download triggered.')


## Step 5 — QLoRA Fine-Tuning *(optional — reproduces Section IV-B)*

**This step is optional.** Reproduces the overfitting finding: training loss collapses near-zero within epoch 1 on 790 samples. This is the **expected failure** — the overfitting *is* the finding.

> *Loss collapsed 0.211 → 6.6×10⁻⁵ within epoch 1. Severe overfitting. Minimum ~5,000 pairs needed for stable 7B VLM fine-tuning.* — Section IV-B

**Runtime:** A100 ~45–60 min | T4 ~3–4 hours


In [ ]:
import shutil, subprocess, sys, os, torch

SCRIPT_SRC = f'{PROJECT_DIR}/training/qlora_finetune.py'
SCRIPT_DST = '/content/qlora_finetune.py'

if not os.path.exists(SCRIPT_SRC):
    print(f'ERROR: {SCRIPT_SRC} not found.')
    print('Ensure training/qlora_finetune.py was in semifa_dataset.tar.gz')
else:
    shutil.copy(SCRIPT_SRC, SCRIPT_DST)
    GPU   = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    BF16  = 'A100' in GPU or 'A6000' in GPU
    BATCH = 2 if BF16 else 1

    cmd = [sys.executable, SCRIPT_DST,
           '--output-dir', QLORA_DIR, '--image-root', PROJECT_DIR,
           '--epochs', '3', '--batch-size', str(BATCH),
           '--grad-accum', '8', '--lora-r', '16']
    if not BF16: cmd.append('--no-bf16')

    env = os.environ.copy()
    env['TRAIN_JSONL'] = TRAIN_JSONL
    env['VAL_JSONL']   = VAL_JSONL

    print(f'GPU: {GPU}  |  BF16: {BF16}  |  Batch: {BATCH}')
    print(f'Output: {QLORA_DIR}')
    print('Expected: loss collapses to <0.001 within epoch 1 (overfitting on 790 samples)\n')

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, env=env)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()

    print(f'\nProcess exited: {proc.returncode}')
    print('Finding: severe overfitting. Base LLaVA-1.6 zero-shot outperforms fine-tuned adapter.')
    print('Minimum dataset for stable QLoRA on 7B VLM: ~5,000 image-QA pairs.')
